# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a")

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "cntry": "country"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Customer ID cleanup

In [0]:
df = (
    df
    .withColumn(
        "customer_number", 
        F.regexp_replace(col("customer_number"), "-", ""))
)

## Country normalization

In [0]:
df = (
    df
    .withColumn(
        "country",
        F.when(col("country") == "DE", "Germany")
         .when(col("country").isin("US", "USA"), "United States")
         .when((col("country") == "") | col("country").isNull(), "n/a")
         .otherwise(col("country"))
    )
)

## Sanity check of final DataFrame

In [0]:
df.limit(10).display()

# Write into silver table

In [0]:
(
    df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("workspace.silver.erp_customer_location")
)